# Article 1 - Leakage-Aware and Cost-Sensitive Machine Learning for Highly Imbalanced Smart Irrigation Decision-Making

**Reproduction notebook.** Random stratified vs date-grouped cross-validation for six cost-sensitive classifiers (Tables I-IV, Figs. 1-2), plus the fold-assignment sensitivity of Sec. IV-D.

**Run time on Colab:** about 10 min for the main tables. The sensitivity analysis (61 fold assignments) adds roughly 30-45 min on a free Colab CPU; set `RUN_SENSITIVITY = False` below to skip it.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badrelkhalylyphd-boop/smart-irrigation-leakage-aware-validation/blob/main/reproduce_colab.ipynb)

### How to use
1. **Runtime > Run all.** The first cell installs the exact library versions used for the paper. On Colab the session then restarts by itself.
2. Upload `greenhouse_dataset.csv` in the **Files** panel of Colab (folder icon on the left; it goes to `/content`). The file is in `data/` of this repository or on [Mendeley Data](https://doi.org/10.17632/2tvpsht4s4.1). It survives the automatic restart of step 1.
3. **Runtime > Run all** a second time. Cell 2 reads `/content/greenhouse_dataset.csv` directly and checks its SHA-256.
4. The last section compares every number of the revised paper with this run and marks each one ✅ or ❌.

Locally (Jupyter), open the notebook from the repository root: the data file is picked up from `data/`.

## 1. Environment

In [ ]:
# Install the exact library versions used for the paper (runs once).
import os, sys, subprocess, importlib.metadata as md
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
PINNED = {'numpy': '2.2.6', 'pandas': '2.3.3', 'scikit-learn': '1.7.2', 'scipy': '1.15.3', 'lightgbm': '4.7.0', 'matplotlib': '3.11.2', 'pyarrow': '20.0.0', 'xgboost': '3.2.0', 'catboost': '1.2.10'}

def _version(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

wrong = {p: _version(p) for p, v in PINNED.items() if _version(p) != v}
if wrong:
    print("Installing pinned versions (currently:", wrong, ") ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + [f"{p}=={v}" for p, v in PINNED.items()])
    if IN_COLAB:
        print("\nDone. The session restarts now. Then choose Runtime > Run all again.")
        os.kill(os.getpid(), 9)
    raise SystemExit("Pinned versions installed. Restart the kernel, then run all cells again.")
print("Environment OK -", sys.version.split()[0], PINNED)


## 2. Data

In [ ]:
# Get the dataset.
# On Colab: upload greenhouse_dataset.csv in the Files panel (left sidebar); it lands in /content and is read from there.
# Locally: the copy in data/ of the repository is used.
import hashlib, shutil, urllib.request
from pathlib import Path

COLAB_FILE = Path("/content/greenhouse_dataset.csv")   # file uploaded through the Colab Files panel
DATA_URL = ""  # optional, e.g. "https://raw.githubusercontent.com/badrelkhalylyphd-boop/smart-irrigation-leakage-aware-validation/main/data/greenhouse_dataset.csv"
EXPECTED_SHA256 = "e64a904a981d323f67210138cfc078e98d1911e6d2bb42195bf76aa818c51aff"

WORK = Path("work").resolve()
SRC, OUT = WORK / "src", WORK / "outputs"
SRC.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
DATA = WORK / "greenhouse_dataset.csv"

candidates = [COLAB_FILE, *sorted(Path("/content").glob("greenhouse_dataset*.csv")),
              Path("data/greenhouse_dataset.csv"), Path("../data/greenhouse_dataset.csv"), Path("greenhouse_dataset.csv")]
found = [p for p in candidates if p.is_file()]
if found:
    shutil.copy(found[0], DATA)
    print("Dataset read from", found[0])
elif DATA.exists():
    print("Dataset already in", DATA)
elif DATA_URL:
    urllib.request.urlretrieve(DATA_URL, DATA)
elif IN_COLAB:
    from google.colab import files
    print("greenhouse_dataset.csv not found in /content. Select it now (or upload it in the Files panel and rerun this cell).")
    name = next(iter(files.upload()))
    Path(name).replace(DATA)
else:
    raise FileNotFoundError("Put greenhouse_dataset.csv in data/ or next to this notebook, or set DATA_URL.")

sha = hashlib.sha256(DATA.read_bytes()).hexdigest()
print("SHA-256:", sha)
print("Same file as in the paper." if sha == EXPECTED_SHA256 else "WARNING: this is not the file used in the paper.")


## 3. Code
The cells below write the scripts of `src/` unchanged into `work/src/`.

In [ ]:
%%writefile work/src/prep1.py
"""Article 1 - preprocessing for leakage-aware smart-irrigation classification.
Usage: python prep1.py greenhouse_dataset.csv
Outputs: irrigation_clean.parquet
"""
import sys
import numpy as np
import pandas as pd

INPUT = sys.argv[1] if len(sys.argv) > 1 else "greenhouse_dataset.csv"
FEATURES = ["Soil Percent", "Soil Raw", "Suhu Tanah", "Suhu Udara", "Humidity"]

# The source CSV is semicolon-separated.
df = pd.read_csv(INPUT, sep=";")
print("Raw:", df.shape)
print("Label counts (including missing):", df["Label"].value_counts(dropna=False).to_dict())

# Article 1 is supervised: retain only rows with an observed irrigation label.
df = df[df["Label"].notna()].copy()
df["Label"] = df["Label"].astype(int)
print("Labeled:", len(df), df["Label"].value_counts().to_dict())

# Convert non-finite sensor measurements to missing.
df[FEATURES] = df[FEATURES].replace([np.inf, -np.inf], np.nan)
missing_sensor = int(df[FEATURES].isna().any(axis=1).sum())
print("Rows with missing/non-finite sensor values:", missing_sensor)

# Two timestamp formats occur in the file:
# 2026-04-23_04-51-49
# 2026-05-25 21:48:31
# Normalize both formats without using future information.
s = df["Timestamp"].astype(str).str.replace("_", " ", regex=False)
parts = s.str.split(" ", n=1, expand=True)
parts[1] = parts[1].str.replace("-", ":", regex=False)
df["Datetime"] = pd.to_datetime(parts[0] + " " + parts[1], errors="coerce")
invalid_time = int(df["Datetime"].isna().sum())
print("Rows with invalid timestamps:", invalid_time)

# Remove unusable supervised records (18 sensor-missing + 2 invalid timestamps in v1).
df = df.dropna(subset=FEATURES + ["Datetime"]).copy()
print("After unusable-row removal:", len(df))

# Exact duplicate feature-label vectors are removed before any validation split.
before = len(df)
df = df.drop_duplicates(subset=FEATURES + ["Label"]).copy()
print("Exact duplicate feature-label rows removed:", before - len(df))

# Safety check: the same feature vector must not carry contradictory labels.
amb = df.groupby(FEATURES, dropna=False)["Label"].nunique()
ambiguous = int((amb > 1).sum())
print("Contradictory feature vectors:", ambiguous)
if ambiguous:
    bad = amb[amb > 1].reset_index()[FEATURES]
    df = df.merge(bad.assign(_bad=1), on=FEATURES, how="left")
    df = df[df["_bad"].isna()].drop(columns="_bad")

# Calendar day is used only as a grouping variable for leakage-aware CV.
df["DateGroup"] = df["Datetime"].dt.strftime("%Y-%m-%d")
print("Final:", len(df), df["Label"].value_counts().to_dict())
print("Positive prevalence (%):", round(100 * df["Label"].mean(), 4))
print("Number of calendar-day groups:", df["DateGroup"].nunique())

df.to_parquet("irrigation_clean.parquet", index=False)


In [ ]:
%%writefile work/src/eval1.py
"""Article 1 - random vs date-grouped leakage-aware evaluation.
Usage: python eval1.py
Requires irrigation_clean.parquet produced by prep1.py.
Outputs: article1_results.csv, article1_oof_predictions.parquet
"""
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, average_precision_score, confusion_matrix
)
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

SEED = 42
FEATURES = ["Soil Percent", "Soil Raw", "Suhu Tanah", "Suhu Udara", "Humidity"]
df = pd.read_parquet("irrigation_clean.parquet")
X = df[FEATURES]
y = df["Label"].to_numpy(dtype=int)
groups = df["DateGroup"].to_numpy()


def build_model(name, y_train):
    """Create a cost-sensitive classifier using training-fold information only."""
    if name == "Logistic Regression":
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED),
        )
    if name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=SEED
        )
    if name == "Extra Trees":
        return ExtraTreesClassifier(
            n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=SEED
        )
    if name == "LightGBM":
        return lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.05, class_weight="balanced",
            random_state=SEED, n_jobs=-1, verbose=-1,
        )
    if name == "XGBoost":
        n_neg = int((y_train == 0).sum())
        n_pos = int((y_train == 1).sum())
        return XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            scale_pos_weight=n_neg / max(n_pos, 1), eval_metric="logloss",
            random_state=SEED, n_jobs=-1,
        )
    if name == "CatBoost":
        return CatBoostClassifier(
            iterations=300, depth=6, learning_rate=0.05,
            auto_class_weights="Balanced", verbose=False, random_seed=SEED,
        )
    raise ValueError(name)


def score(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "BalancedAccuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "PRAUC": average_precision_score(y_true, y_prob),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
    }


def evaluate(protocol, splitter):
    rows = []
    oof_rows = []
    names = ["Logistic Regression", "Random Forest", "Extra Trees", "LightGBM", "XGBoost", "CatBoost"]
    split_iter = splitter.split(X, y, groups) if protocol == "Date-grouped" else splitter.split(X, y)
    splits = list(split_iter)

    # Group-leakage assertion for the grouped protocol.
    if protocol == "Date-grouped":
        for tr, va in splits:
            assert set(groups[tr]).isdisjoint(set(groups[va]))

    for name in names:
        pred = np.zeros(len(y), dtype=int)
        prob = np.zeros(len(y), dtype=float)
        fold_id = np.full(len(y), -1, dtype=int)

        for fold, (tr, va) in enumerate(splits):
            model = build_model(name, y[tr])
            model.fit(X.iloc[tr], y[tr])
            pred[va] = np.asarray(model.predict(X.iloc[va])).reshape(-1).astype(int)
            prob[va] = model.predict_proba(X.iloc[va])[:, 1]
            fold_id[va] = fold

        m = score(y, pred, prob)
        rows.append({"Protocol": protocol, "Model": name, **m})
        oof_rows.append(pd.DataFrame({
            "Protocol": protocol, "Model": name, "Row": np.arange(len(y)),
            "Fold": fold_id, "True": y, "Pred": pred, "Prob": prob,
            "DateGroup": groups,
        }))
        print(protocol, name, {k: round(100*v, 2) if k not in {"TP", "TN", "FP", "FN"} else v
                               for k, v in m.items()})
    return rows, oof_rows


random_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
group_cv = StratifiedGroupKFold(n_splits=5)  # no shuffle: deterministic, as used for the reported results

rows_a, oof_a = evaluate("Random stratified", random_cv)
rows_b, oof_b = evaluate("Date-grouped", group_cv)

res = pd.DataFrame(rows_a + rows_b)
res.to_csv("article1_results.csv", index=False)
pd.concat(oof_a + oof_b, ignore_index=True).to_parquet("article1_oof_predictions.parquet", index=False)
print("\nSaved article1_results.csv and article1_oof_predictions.parquet")


In [ ]:
%%writefile work/src/figs1.py
"""Article 1 - generate manuscript tables/figures from article1_results.csv.
Usage: python figs1.py
"""
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

r = pd.read_csv("article1_results.csv")
metrics = ["Accuracy", "BalancedAccuracy", "Precision", "Recall", "F1", "MCC", "PRAUC"]
print("\nResults (%)")
print(r[["Protocol", "Model"] + metrics].assign(**{m: r[m] * 100 for m in metrics}).round(2).to_string(index=False))

# F1 comparison
p = r.pivot(index="Model", columns="Protocol", values="F1") * 100
ax = p.plot(kind="bar", figsize=(8, 4))
ax.set_ylabel("F1-score (%)")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("fig_f1_random_vs_grouped.png", dpi=300)
plt.close()

# PR-AUC comparison
p = r.pivot(index="Model", columns="Protocol", values="PRAUC") * 100
ax = p.plot(kind="bar", figsize=(8, 4))
ax.set_ylabel("PR-AUC (%)")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("fig_prauc_random_vs_grouped.png", dpi=300)
plt.close()

# Validation-gap table
q = r.pivot(index="Model", columns="Protocol", values="F1") * 100
q["F1_drop_points"] = q["Random stratified"] - q["Date-grouped"]
q.to_csv("table_f1_validation_gap.csv")
print("\nF1 degradation (percentage points)")
print(q[["F1_drop_points"]].round(2).to_string())


In [ ]:
%%writefile work/src/fold_sensitivity1.py
"""Article 1 - sensitivity of the date-grouped estimate to the fold assignment.
Usage: python fold_sensitivity1.py
Requires irrigation_clean.parquet produced by prep1.py.
Outputs: article1_fold_sensitivity.csv

With 17 calendar days and positives concentrated on a few of them, the way days
are packed into five folds changes the grouped score. This script re-runs the
grouped protocol for the deterministic split used in the paper and for 60
shuffled splits (seeds 0-59), keeping only distinct day-to-fold assignments.
"""
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, matthews_corrcoef, average_precision_score
import lightgbm as lgb

SEED = 42
FEATURES = ["Soil Percent", "Soil Raw", "Suhu Tanah", "Suhu Udara", "Humidity"]
df = pd.read_parquet("irrigation_clean.parquet")
X = df[FEATURES]
y = df["Label"].to_numpy(dtype=int)
groups = df["DateGroup"].to_numpy()

MODELS = {
    "Random Forest": lambda: RandomForestClassifier(
        n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=SEED),
    "LightGBM": lambda: lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.05, class_weight="balanced",
        random_state=SEED, n_jobs=-1, verbose=-1),
}

def run(splits, make_model):
    pred = np.zeros(len(y), dtype=int)
    prob = np.zeros(len(y), dtype=float)
    for tr, va in splits:
        m = make_model().fit(X.iloc[tr], y[tr])
        pred[va] = m.predict(X.iloc[va])
        prob[va] = m.predict_proba(X.iloc[va])[:, 1]
    return f1_score(y, pred), matthews_corrcoef(y, pred), average_precision_score(y, prob)

configs = [("no-shuffle (paper)", StratifiedGroupKFold(n_splits=5))]
configs += [(f"shuffle seed {s}", StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=s))
            for s in range(60)]

rows, seen = [], set()
for label, cv in configs:
    splits = list(cv.split(X, y, groups))
    key = tuple(sorted(tuple(sorted(set(groups[va]))) for _, va in splits))
    if key in seen:  # identical day-to-fold packing already evaluated
        continue
    seen.add(key)
    for name, make in MODELS.items():
        f1, mcc, ap = run(splits, make)
        rows.append({"Split": label, "Model": name, "F1": f1, "MCC": mcc, "PRAUC": ap})
    print(label, "done", flush=True)

res = pd.DataFrame(rows)
res.to_csv("article1_fold_sensitivity.csv", index=False)
summary = (res.groupby("Model")[["F1", "MCC", "PRAUC"]]
              .agg(["mean", "std", "min", "max"]) * 100).round(2)
print("\nDistinct fold assignments evaluated:", len(seen))
print(summary.to_string())


In [ ]:
# Helper: run a script from work/src inside work/outputs and stream its output.
def run(script, *args):
    cmd = [sys.executable, "-u", str(SRC / script), *map(str, args)]
    env = dict(os.environ, MPLBACKEND="Agg", PYTHONIOENCODING="utf-8")
    p = subprocess.Popen(cmd, cwd=OUT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        if "Warning" not in line and "warnings.warn" not in line:
            print(line, end="")
    if p.wait():
        raise RuntimeError(f"{script} failed")


## 4. Run the experiment
All outputs go to `work/outputs/`.

### Step 1 - cleaning (Table I)

In [ ]:
run("prep1.py", DATA)

### Step 2 - random vs date-grouped CV (Tables II-III)

In [ ]:
run("eval1.py")

### Step 3 - validation gap and figures (Table IV, Figs. 1-2)

In [ ]:
run("figs1.py")

### Step 4 - sensitivity to the fold assignment (Sec. IV-D)

In [ ]:
RUN_SENSITIVITY = True
if RUN_SENSITIVITY:
    run("fold_sensitivity1.py")
else:
    print("Skipped.")

## 5. Check against the revised paper

In [ ]:
# Compare every reported number with what this run produced.
import math
import pandas as pd
CHECKS = []

def chk(item, expected, obtained, tol=0.006):
    """Numbers are compared at the precision printed in the paper."""
    if isinstance(expected, str):
        ok = str(obtained) == expected
    else:
        ok = obtained is not None and not (isinstance(obtained, float) and math.isnan(obtained)) and abs(obtained - expected) <= tol
    CHECKS.append({"item": item, "paper": expected,
                   "this run": round(obtained, 4) if isinstance(obtained, float) else obtained,
                   "status": "✅" if ok else "❌"})

r = pd.read_csv(OUT / "article1_results.csv").set_index(["Protocol", "Model"])
clean = pd.read_parquet(OUT / "irrigation_clean.parquet")
chk("Table I | unique labeled rows", 48642, len(clean), 0)
chk("Table I | irrigation events", 208, int(clean.Label.sum()), 0)
chk("Table I | calendar days", 17, clean.DateGroup.nunique(), 0)

cols = ["Accuracy", "BalancedAccuracy", "Precision", "Recall", "F1", "MCC", "PRAUC"]
T2 = {"Logistic Regression": (97.54, 98.05, 14.64, 98.56, 25.50, 37.50, 56.31),
      "Random Forest": (99.93, 92.79, 98.34, 85.58, 91.52, 91.71, 95.91),
      "Extra Trees": (99.94, 93.27, 98.36, 86.54, 92.07, 92.23, 95.34),
      "LightGBM": (99.94, 96.14, 94.58, 92.31, 93.43, 93.41, 94.84),
      "XGBoost": (99.94, 96.38, 93.69, 92.79, 93.24, 93.21, 96.27),
      "CatBoost": (99.91, 97.08, 85.22, 94.23, 89.50, 89.56, 96.91)}
T3 = {"Logistic Regression": (95.47, 96.29, 8.42, 97.12, 15.50, 27.90, 20.93),
      "Random Forest": (99.61, 89.99, 53.18, 80.29, 63.98, 65.17, 70.61),
      "Extra Trees": (99.61, 85.68, 52.84, 71.63, 60.82, 61.33, 72.18),
      "LightGBM": (99.17, 95.03, 32.87, 90.87, 48.28, 54.38, 79.47),
      "XGBoost": (99.30, 95.34, 37.11, 91.35, 52.78, 57.98, 76.96),
      "CatBoost": (99.06, 95.94, 30.49, 92.79, 45.90, 52.90, 81.52)}
for proto, tab, name in (("Random stratified", T2, "Table II"), ("Date-grouped", T3, "Table III")):
    for model, vals in tab.items():
        for c, v in zip(cols, vals):
            chk(f"{name} | {model} | {c}", v, 100 * r.loc[(proto, model), c])

T4 = {"Random Forest": 27.53, "Extra Trees": 31.26, "LightGBM": 45.15, "XGBoost": 40.46, "CatBoost": 43.60}
for model, drop in T4.items():
    got = 100 * (r.loc[("Random stratified", model), "F1"] - r.loc[("Date-grouped", model), "F1"])
    chk(f"Table IV | {model} | F1 drop (points)", drop, got)

sens = OUT / "article1_fold_sensitivity.csv"
if sens.exists():
    s = pd.read_csv(sens)
    chk("Sec. IV-D | distinct fold assignments", 61, s.Split.nunique(), 0)
    for model, (mean, sd, lo, hi) in {"LightGBM": (55.30, 5.36, 40.33, 65.28),
                                      "Random Forest": (59.65, 21.42, 0.81, 76.22)}.items():
        f = 100 * s[s.Model == model].F1
        chk(f"Sec. IV-D | {model} | mean F1", mean, f.mean()); chk(f"Sec. IV-D | {model} | SD F1", sd, f.std())
        chk(f"Sec. IV-D | {model} | min F1", lo, f.min()); chk(f"Sec. IV-D | {model} | max F1", hi, f.max())
    chk("Sec. IV-D | Random Forest | assignments with F1 < 30%", 11, int((s[s.Model == "Random Forest"].F1 < .30).sum()), 0)


In [ ]:
report = pd.DataFrame(CHECKS)
n_ok = (report.status == "✅").sum()
print(f"{n_ok} / {len(report)} values match the revised paper.")
pd.set_option("display.max_rows", 500)
report[report.status == "❌"] if n_ok < len(report) else print("All checks passed.")


Full list of checks:

In [ ]:
report

## 6. Figures

In [ ]:
# Figures produced by this run
from IPython.display import Image, display
for f in sorted(OUT.rglob("*.png")):
    print(f.name); display(Image(filename=str(f), width=520))
